# HMM Grid Search For Selected Features

Notebook loads the train split from S3, uses the last year before the test cutoff, fits Gaussian HMM models over the requested grid, and stores all results in a dedicated S3 prefix.

In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

load_dotenv(Path.cwd() / '.env')

FEATURES = [
    'ada_kaufman_efficiency_30m',
    'ada_realized_volatility_30m',
    'ada_aggression_delta_norm_10m',
    'ada_volume_entropy_norm_10m',
    'ada_pressure_concentration_10m',
    'ada_minus_btc_log_return_30m',
]

N_COMPONENTS_GRID = [2, 3, 4, 5, 6, 7, 8]
COVARIANCE_TYPE_GRID = ['diag', 'full']
RANDOM_STATE_GRID = list(range(10))

N_ITER = 1000
TOL = 1e-4
MIN_COVAR = 1e-3
TEST_CUTOFF = pd.Timestamp('2025-02-01T00:00:00Z')
TRAIN_START = TEST_CUTOFF - pd.DateOffset(years=1)
DEGENERATE_STATE_SHARE_THRESHOLD = 0.01

BUCKET = os.getenv('YC_BUCKET', 'binance-data-downloader')
TRAIN_KEY = 'features/hmm_dataset/splits/hmm_dataset_train.parquet'
RESULT_PREFIX = 'analysis/hmm_grid_search_selected_features_v1'

required_env = ['YC_ENDPOINT', 'YC_REGION', 'YC_ACCESS_KEY_ID', 'YC_SECRET_ACCESS_KEY']
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing S3 environment variables: {missing}')

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('YC_ENDPOINT'),
    region_name=os.getenv('YC_REGION'),
    aws_access_key_id=os.getenv('YC_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('YC_SECRET_ACCESS_KEY'),
)

print(f'Train window: {TRAIN_START} <= timestamp < {TEST_CUTOFF}')
print(f'Input: s3://{BUCKET}/{TRAIN_KEY}')
print(f'Results: s3://{BUCKET}/{RESULT_PREFIX}/')

Train window: 2024-02-01 00:00:00+00:00 <= timestamp < 2025-02-01 00:00:00+00:00
Input: s3://binance-data-downloader/features/hmm_dataset/splits/hmm_dataset_train.parquet
Results: s3://binance-data-downloader/analysis/hmm_grid_search_selected_features_v1/


In [2]:
def hmm_parameter_count(n_components: int, n_features: int, covariance_type: str) -> int:
    startprob_params = n_components - 1
    transmat_params = n_components * (n_components - 1)
    mean_params = n_components * n_features
    if covariance_type == 'diag':
        covariance_params = n_components * n_features
    elif covariance_type == 'full':
        covariance_params = n_components * n_features * (n_features + 1) // 2
    else:
        raise ValueError(f'Unsupported covariance_type: {covariance_type}')
    return startprob_params + transmat_params + mean_params + covariance_params


def sequence_run_metrics(states: np.ndarray, n_components: int) -> tuple[float, dict[str, float]]:
    if len(states) == 0:
        return math.nan, {}
    change_positions = np.flatnonzero(np.diff(states) != 0) + 1
    starts = np.r_[0, change_positions]
    ends = np.r_[change_positions, len(states)]
    run_states = states[starts]
    run_lengths = ends - starts
    metrics = {}
    for state in range(n_components):
        state_lengths = run_lengths[run_states == state]
        metrics[f'state_{state}_mean_duration'] = float(state_lengths.mean()) if len(state_lengths) else 0.0
        metrics[f'state_{state}_run_count'] = int(len(state_lengths))
    return float(run_lengths.mean()), metrics


def theoretical_mean_duration(transmat: np.ndarray, shares: np.ndarray) -> float:
    diagonal = np.clip(np.diag(transmat), 0.0, 0.999999)
    durations = 1.0 / (1.0 - diagonal)
    return float(np.sum(shares * durations))


def upload_file(local_path: Path, key: str, content_type: str | None = None) -> None:
    extra_args = {}
    if content_type:
        extra_args['ContentType'] = content_type
    s3.upload_file(str(local_path), BUCKET, key, ExtraArgs=extra_args or None)
    head = s3.head_object(Bucket=BUCKET, Key=key)
    print(f'Uploaded s3://{BUCKET}/{key} ({head["ContentLength"] / 1024**2:.2f} MiB)')

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix='hmm_grid_search_'))
train_path = workdir / 'hmm_dataset_train.parquet'
print(f'Downloading to {train_path}')
s3.download_file(BUCKET, TRAIN_KEY, str(train_path))

columns = ['timestamp', *FEATURES]
data = pd.read_parquet(train_path, columns=columns)
data['timestamp'] = pd.to_datetime(data['timestamp'], utc=True)
data = data.loc[data['timestamp'].ge(TRAIN_START) & data['timestamp'].lt(TEST_CUTOFF)].copy()
data = data.sort_values('timestamp').dropna(subset=FEATURES).reset_index(drop=True)

for feature in FEATURES:
    data[feature] = pd.to_numeric(data[feature], errors='raise').astype('float64')

values = data[FEATURES].to_numpy(dtype='float64')
if not np.isfinite(values).all():
    raise ValueError('Training window contains non-finite feature values')

scaler = StandardScaler()
X = scaler.fit_transform(values).astype('float64')
n_observations, n_features = X.shape

print(f'Rows: {n_observations:,}; features: {n_features}')
print(f'Timestamp range: {data["timestamp"].min()} -> {data["timestamp"].max()}')

In [ ]:
records = []
best_model = None
best_record = None
total = len(N_COMPONENTS_GRID) * len(COVARIANCE_TYPE_GRID) * len(RANDOM_STATE_GRID)
started_at = datetime.now(timezone.utc)

for idx, (n_components, covariance_type, random_state) in enumerate(
    (params for n in N_COMPONENTS_GRID for params in [(n, c, r) for c in COVARIANCE_TYPE_GRID for r in RANDOM_STATE_GRID]),
    start=1,
):
    label = f'[{idx}/{total}] n={n_components} cov={covariance_type} seed={random_state}'
    print(label, flush=True)
    model = GaussianHMM(
        n_components=n_components,
        covariance_type=covariance_type,
        n_iter=N_ITER,
        tol=TOL,
        min_covar=MIN_COVAR,
        random_state=random_state,
        verbose=False,
    )
    record = {
        'n_components': n_components,
        'covariance_type': covariance_type,
        'random_state': random_state,
        'n_iter': N_ITER,
        'tol': TOL,
        'min_covar': MIN_COVAR,
        'n_observations': n_observations,
        'n_features': n_features,
        'status': 'ok',
    }
    try:
        model.fit(X)
        log_likelihood = float(model.score(X))
        n_parameters = hmm_parameter_count(n_components, n_features, covariance_type)
        aic = float(2 * n_parameters - 2 * log_likelihood)
        bic = float(math.log(n_observations) * n_parameters - 2 * log_likelihood)
        states = model.predict(X)
        counts = np.bincount(states, minlength=n_components)
        shares = counts / counts.sum()
        avg_duration, duration_metrics = sequence_run_metrics(states, n_components)

        record.update(
            {
                'log_likelihood': log_likelihood,
                'aic': aic,
                'bic': bic,
                'n_parameters': n_parameters,
                'converged': bool(model.monitor_.converged),
                'em_iterations': int(model.monitor_.iter),
                'average_sequence_duration': avg_duration,
                'average_theoretical_duration': theoretical_mean_duration(model.transmat_, shares),
                'min_state_share': float(shares.min()),
                'has_degenerate_state': bool(shares.min() < DEGENERATE_STATE_SHARE_THRESHOLD),
            }
        )
        for state in range(n_components):
            record[f'state_{state}_share'] = float(shares[state])
            record[f'state_{state}_count'] = int(counts[state])
            record[f'state_{state}_self_transition'] = float(model.transmat_[state, state])
        record.update(duration_metrics)

        if best_record is None or bic < best_record['bic']:
            best_record = dict(record)
            best_model = model
    except Exception as exc:
        record.update({'status': 'failed', 'error': repr(exc)})
        print(f'  failed: {exc}', flush=True)
    finally:
        records.append(record)
        if best_model is not model:
            del model
        gc.collect()

finished_at = datetime.now(timezone.utc)
metrics = pd.DataFrame(records).sort_values(['status', 'bic'], na_position='last').reset_index(drop=True)
metrics.head(20)

In [ ]:
if best_model is None or best_record is None:
    raise RuntimeError('No HMM model finished successfully')

best_states = best_model.predict(X)
state_profiles = []
for state in range(best_model.n_components):
    mask = best_states == state
    row = {
        'state': state,
        'count': int(mask.sum()),
        'share': float(mask.mean()),
        'self_transition': float(best_model.transmat_[state, state]),
    }
    for feature in FEATURES:
        row[f'{feature}_mean'] = float(data.loc[mask, feature].mean())
        row[f'{feature}_std'] = float(data.loc[mask, feature].std(ddof=1))
    state_profiles.append(row)

states_frame = pd.DataFrame({'timestamp': data['timestamp'], 'state': best_states.astype('int16')})
profiles = pd.DataFrame(state_profiles)

near_best = metrics.loc[metrics['status'].eq('ok')].copy()
near_best = near_best.loc[near_best['bic'] <= best_record['bic'] * 1.01]
near_best = near_best.sort_values(
    ['has_degenerate_state', 'average_sequence_duration', 'bic'],
    ascending=[True, False, True],
).head(20)

summary = {
    'started_at': started_at.isoformat(),
    'finished_at': finished_at.isoformat(),
    'bucket': BUCKET,
    'input_key': TRAIN_KEY,
    'result_prefix': RESULT_PREFIX,
    'train_start': TRAIN_START.isoformat(),
    'test_cutoff': TEST_CUTOFF.isoformat(),
    'features': FEATURES,
    'grid': {
        'n_components': N_COMPONENTS_GRID,
        'covariance_type': COVARIANCE_TYPE_GRID,
        'random_state': RANDOM_STATE_GRID,
        'n_iter': N_ITER,
        'tol': TOL,
        'min_covar': MIN_COVAR,
    },
    'selection_rule': 'Primary criterion is minimum BIC. Near-BIC candidates are listed for interpretability review.',
    'degenerate_state_share_threshold': DEGENERATE_STATE_SHARE_THRESHOLD,
    'best_model': best_record,
    'near_best_candidates': near_best.to_dict(orient='records'),
}

print(json.dumps(summary['best_model'], indent=2))

In [ ]:
run_dir = workdir / 'results'
run_dir.mkdir(exist_ok=True)

metrics_path = run_dir / 'grid_search_metrics.parquet'
metrics_csv_path = run_dir / 'grid_search_metrics.csv'
profiles_path = run_dir / 'best_model_state_profiles.parquet'
states_path = run_dir / 'best_model_train_states.parquet'
bundle_path = run_dir / 'best_model_bundle.joblib'
summary_path = run_dir / 'summary.json'
params_path = run_dir / 'run_config.json'

metrics.to_parquet(metrics_path, index=False, compression='zstd')
metrics.to_csv(metrics_csv_path, index=False)
profiles.to_parquet(profiles_path, index=False, compression='zstd')
states_frame.to_parquet(states_path, index=False, compression='zstd')
joblib.dump({'model': best_model, 'scaler': scaler, 'features': FEATURES, 'best_record': best_record}, bundle_path)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
params_path.write_text(json.dumps(summary['grid'], indent=2), encoding='utf-8')

artifacts = [
    (metrics_path, 'grid_search_metrics.parquet', 'application/vnd.apache.parquet'),
    (metrics_csv_path, 'grid_search_metrics.csv', 'text/csv'),
    (profiles_path, 'best_model_state_profiles.parquet', 'application/vnd.apache.parquet'),
    (states_path, 'best_model_train_states.parquet', 'application/vnd.apache.parquet'),
    (bundle_path, 'best_model_bundle.joblib', 'application/octet-stream'),
    (summary_path, 'summary.json', 'application/json'),
    (params_path, 'run_config.json', 'application/json'),
]

manifest = []
for local_path, name, content_type in artifacts:
    key = f'{RESULT_PREFIX}/{name}'
    upload_file(local_path, key, content_type)
    manifest.append({'name': name, 's3_key': key, 'size_bytes': local_path.stat().st_size})

manifest_path = run_dir / 'manifest.json'
manifest_path.write_text(json.dumps({'artifacts': manifest}, indent=2), encoding='utf-8')
upload_file(manifest_path, f'{RESULT_PREFIX}/manifest.json', 'application/json')

print(f'Done: s3://{BUCKET}/{RESULT_PREFIX}/')